# User-defined local and global equations

A quadratic Darcy pressure $p=1+x^2+y^2$ gives physical flux $q=-2(x,y)$ and independently differentiated source $f=-4$. P2 locals and P0 flux traces on two macrotriangles give a small exactly representable analytical check.

The user supplies four blocks; the library eliminates the local complement:

$$
\begin{aligned}
A_T u_T+B_T\lambda_T &= f_T,\\
C_Tu_T+D_T\lambda_T &= g_T.
\end{aligned}
$$

Here $a_T(p,v)=(\nabla p,\nabla v)_T$, $L_T(v)=(-4,v)_T$,
$B_T\lambda=\langle\lambda,v\rangle_{\partial T}$ and
$C_Tp=-\langle p,\mu\rangle_{\partial T}$ with signed macroface normals.
$D_T=0$ and $g_T=0$. The constant local kernel is retained globally;
the physical integral selects its complement. The global right-hand side is
$-\langle p_D,\mu\rangle_{\partial\Omega}$ in trace coordinates, followed
by zeros in retained-mode coordinates. No model name chooses these forms.


In [ ]:
from pathlib import Path
import sys
import numpy as np

ROOT = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "pyproject.toml").is_file()
    and (candidate / "src/pymhm").is_dir()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [ ]:
from pymhm import Equation, LocalEquations, MultiscaleProblem, assemble, columns, rows
from pymhm.meshes.triangle import TriangleMesh
from pymhm.fem.traces.interval import SkeletonSpace
from pymhm.fem.scalar.operators import boundary_data
from pymhm.fem.scalar.triangle import scalar_operators, trace_coupling
from examples.tutorial_local_provider import (
    exact_pressure, exact_flux, source, TutorialResult, diagnostics,
)

mesh = TriangleMesh.unit_square()
skeleton = SkeletonSpace(mesh)
local_degree, local_refinement = 2, 2

def local_equations(cell):
    """Write local pressure equations and the independent global balance rows."""
    fine = mesh.submesh(cell, local_refinement)
    a, mass, L = scalar_operators(
        fine, local_degree, diffusion=1.0, source=source, order=4,
    )
    b = trace_coupling(mesh, cell, fine, skeleton, local_degree)
    kernel = np.ones((a.shape[0], 1))
    physical_mean = mass @ kernel
    return LocalEquations(
        a, L, columns(*b.T), rows(*(-b.T)), skeleton.cell_dofs(cell),
        kernel=kernel, moments=physical_mean / (kernel.T @ physical_mean).item(),
        metadata=(fine, physical_mean[:, 0]),
    )

boundary, fixed = boundary_data(skeleton, exact_pressure, order=4)
global_equation = Equation(0, -np.r_[boundary, np.zeros(len(mesh.cells))])
problem = MultiscaleProblem(
    global_equation, local_equations, range(len(mesh.cells)), skeleton.size,
    (1,) * len(mesh.cells), fixed=fixed,
)
system = assemble(problem)
solution = system.solve()
result = TutorialResult(problem, system, solution, "primal", "dirichlet", "basix")
record = diagnostics(result)
print(record)
assert max(record["field_L2_errors"].values()) < 1e-10


## Mixed locals, boundary data and an external local response

The importable application provider writes the same four blocks for primal P2 or mixed RT0/P0 locals, including local boundary-pressure multipliers. The mixed third equation prescribes normal flux; omitting it changes the method. Pure Neumann data use the physical mean $5/3$. The RT0 pressure is a P0 projection of the quadratic field. An independent dense response can solve the augmented local columns through the same checked interface.


In [ ]:
from examples.tutorial_local_provider import run_tutorial
from pymhm import ExecutionConfig

execution = ExecutionConfig("serial", native_threads=1, batch_size=1)
records = [
    diagnostics(run_tutorial(formulation=method, boundary=boundary,
                            execution=execution, local_solver="external"))
    for method in ("primal", "mixed")
    for boundary in ("dirichlet", "neumann")
]
for item in records:
    print(item)


## Ordered process batches

Spawn workers use the importable application provider. Each worker supplies oriented numerical responses; the coordinator reduces shared faces in the fixed macrocell order. Notebook-only functions belong to serial/thread execution; process execution requires importable callables.


In [ ]:
parallel = ExecutionConfig("process", workers=2, native_threads=1, batch_size=2)
parallel_record = diagnostics(run_tutorial(execution=parallel, local_solver="external"))
print(parallel_record)
